In [1]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import find_peaks  # (not required for core logic, kept for parity)
import scipy.io as sio
from collections import defaultdict
from kilosort.io import load_ops
from kilosort.data_tools import (
    mean_waveform, cluster_templates, get_cluster_spikes,
    get_spike_waveforms, get_best_channel
)

# ---------------------- waveform analysis ----------------------

def analyze_waveform(waveform, fs):
    """
    Peak-first analysis:
      1) find the global PEAK (max)
      2) find the TROUGh AFTER that peak (min over waveform[max_idx:])
      3) return indices/times/values and Δt (Peak→Trough) in microseconds.
    If no trough exists after peak (peak is last sample), fallback to global trough.
    """
    peak_idx = int(np.argmax(waveform))
    if peak_idx < waveform.size - 1:
        rel_trough = int(np.argmin(waveform[peak_idx:]))
        trough_idx = peak_idx + rel_trough
    else:
        trough_idx = int(np.argmin(waveform))  # fallback

    dt_us = (trough_idx - peak_idx) / fs * 1e6  # microseconds
    return {
        "peak_idx": peak_idx,
        "trough_idx": trough_idx,
        "peak_uV": float(waveform[peak_idx]),
        "trough_uV": float(waveform[trough_idx]),
        "peak_time_us": peak_idx / fs * 1e6,
        "trough_time_us": trough_idx / fs * 1e6,
        "peak_to_trough_us": dt_us
    }

# ---------------------- merged figure (waveform + ISI) ----------------------

def waveform_isi_fig(
    t_us, mean_wv_uV, mean_temp_uV, fs, cluster_id, session_name,
    best_chan, cluster_spikes, all_spikes, save_path_png
):
    """
    Builds a single figure:
      - Top: waveform (dashed, µV vs µs) + template; marks Peak (blue) then Trough (red),
              draws dotted vlines at both, a horizontal connector line between them,
              labels Δt in µs. Title shows best channel, avg FR (whole recording), spike count.
      - Bottom: ISI histogram up to 1 s (x in µs, 10 ms bins), marks the modal bin with a vline + annotation.
    Returns a dict of metrics for Excel.
    """
    # Waveform metrics (Peak → Trough)
    wf = analyze_waveform(mean_wv_uV, fs)
    peak_idx = wf["peak_idx"]
    trough_idx = wf["trough_idx"]
    dt_us = wf["peak_to_trough_us"]

    # Average firing rate using the WHOLE recording duration
    n_spikes = int(cluster_spikes.size)
    rec_duration_s = all_spikes.max() / fs if all_spikes.size > 0 else np.nan
    fr_hz = n_spikes / rec_duration_s if rec_duration_s and rec_duration_s > 0 else float('nan')

    # ISI histogram (0–1 s) in microseconds, 10 ms (10,000 µs) bins
    isi_us = np.diff(cluster_spikes) / fs * 1e6
    bins_us = np.arange(0, 1_000_000 + 10_000, 10_000)  # 0..1,000,000 µs step 10,000 µs
    counts, edges = np.histogram(isi_us, bins=bins_us)
    if counts.size > 0:
        isi_peak_bin = int(np.argmax(counts))
        isi_peak_center_us = (edges[isi_peak_bin] + edges[isi_peak_bin + 1]) / 2.0
        isi_peak_count = int(counts[isi_peak_bin])
    else:
        isi_peak_center_us = np.nan
        isi_peak_count = 0

    # ---- Figure ----
    fig = plt.figure(figsize=(8, 6))
    gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[3, 2], hspace=0.35)

    # Top: Waveform
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(t_us, mean_wv_uV, c='black', linestyle='dashed', linewidth=2, label='Mean waveform')
    ax1.plot(t_us[:mean_temp_uV.shape[-1]], mean_temp_uV, linewidth=1, label='Template')

    # Mark Peak/Trough
    ax1.scatter(t_us[peak_idx], mean_wv_uV[peak_idx], c='blue', zorder=5, label='Peak')
    ax1.scatter(t_us[trough_idx], mean_wv_uV[trough_idx], c='red', zorder=5, label='Trough')

    # Orthogonal lines
    y_min = min(mean_wv_uV.min(), mean_temp_uV.min())
    y_max = max(mean_wv_uV.max(), mean_temp_uV.max())
    ax1.vlines([t_us[peak_idx], t_us[trough_idx]], ymin=y_min, ymax=y_max,
               linestyles='dotted', colors=['blue', 'red'], alpha=0.6)
    y_mid = (mean_wv_uV[peak_idx] + mean_wv_uV[trough_idx]) / 2.0
    ax1.hlines(y=y_mid, xmin=t_us[peak_idx], xmax=t_us[trough_idx],
               linestyles='solid', colors='purple', alpha=0.85)

    # Δt label
    ax1.text((t_us[peak_idx] + t_us[trough_idx]) / 2.0, y_mid,
             f"Δt = {dt_us:.1f} µs", ha='center', va='bottom', fontsize=9, color='purple')

    # Title + labels (units µs, µV)
    title = (f"{session_name} | Cluster {cluster_id} | Best ch {best_chan} | "
             f"FR {fr_hz:.2f} Hz | n={n_spikes}")
    ax1.set_title(title)
    ax1.set_xlabel("Time (µs)")
    ax1.set_ylabel("Voltage (µV)")
    ax1.legend(loc='best', fontsize=8)

    # Bottom: ISI
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.hist(isi_us, bins=bins_us, edgecolor='none')
    ax2.set_title("ISI Histogram (10 ms bins, up to 1 s)")
    ax2.set_xlabel("ISI (µs)")
    ax2.set_ylabel("Count")
    if np.isfinite(isi_peak_center_us):
        ax2.axvline(isi_peak_center_us, linestyle='--', color='k', alpha=0.8)
        ax2.text(isi_peak_center_us, ax2.get_ylim()[1]*0.9,
                 f"Peak ≈ {int(isi_peak_center_us)} µs\nCount={isi_peak_count}",
                 ha='center', va='top', fontsize=8,
                 bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.75))

    fig.savefig(save_path_png, dpi=150, bbox_inches='tight')
    plt.close(fig)

    return {
        "fr_hz": fr_hz,
        "n_spikes": n_spikes,
        "isi_peak_center_us": isi_peak_center_us,
        "isi_peak_count": isi_peak_count,
        "peak_time_us": wf["peak_time_us"],
        "trough_time_us": wf["trough_time_us"],
        "peak_to_trough_us": wf["peak_to_trough_us"],
        "peak_uV": wf["peak_uV"],
        "trough_uV": wf["trough_uV"]
    }

# ---------------------- per-cluster processing ----------------------

def process_cluster(cluster_id, session_name, results_dir, fs_default=30000):
    results_dir = Path(results_dir)

    # Per-session figures folder inside the session path
    figures_dir = results_dir / "figures" / session_name
    figures_dir.mkdir(parents=True, exist_ok=True)

    # Load ops and sample rate
    ops = load_ops(results_dir / 'ops.npy')
    fs = float(ops.get('fs', fs_default))

    # Waveforms & template on best channel
    mean_wv, spike_subset = mean_waveform(cluster_id, results_dir, n_spikes=100, best=True)
    mean_temp = cluster_templates(cluster_id, results_dir, mean=True, best=True, spike_subset=spike_subset)
    best_chan = get_best_channel(results_dir, cluster_id) + 1  # 1-based index for display

    # Time axis in microseconds (match waveform length)
    t_us = (np.arange(mean_wv.shape[-1]) / fs) * 1e6

    # Spikes for this cluster and all spikes for duration
    spikes = np.load(results_dir / 'spike_times.npy')
    cluster_labels = np.load(results_dir / 'spike_clusters.npy')
    cluster_spikes = np.sort(spikes[cluster_labels == cluster_id])

    # Build merged figure and collect metrics
    fig_path = figures_dir / f"{session_name}_cluster{cluster_id}.png"
    fig_metrics = waveform_isi_fig(
        t_us=t_us,
        mean_wv_uV=mean_wv,
        mean_temp_uV=mean_temp,
        fs=fs,
        cluster_id=cluster_id,
        session_name=session_name,
        best_chan=best_chan,
        cluster_spikes=cluster_spikes,
        all_spikes=spikes,
        save_path_png=fig_path
    )

    # Save timestamps in MATLAB .mat
    ts_dir = results_dir / "timestamps"
    ts_dir.mkdir(exist_ok=True)
    mat_filename = ts_dir / f"{cluster_id}.mat"
    sio.savemat(mat_filename, {f"cluster_{cluster_id}_timestamp": cluster_spikes[:, np.newaxis]})

    # Return summary for Excel / best_channels
    return {
        "session": session_name,
        "cluster_id": int(cluster_id),
        "best_channel": int(best_chan),
        "peak_time_us": float(fig_metrics["peak_time_us"]),
        "trough_time_us": float(fig_metrics["trrough_time_us"]) if "trrough_time_us" in fig_metrics else float(fig_metrics["trough_time_us"]),
        "peak_to_trough_us": float(fig_metrics["peak_to_trough_us"]),
        "peak_uV": float(fig_metrics["peak_uV"]),
        "trough_uV": float(fig_metrics["trough_uV"]),
        "isi_peak_center_us": float(fig_metrics["isi_peak_center_us"]) if np.isfinite(fig_metrics["isi_peak_center_us"]) else np.nan,
        "isi_peak_count": int(fig_metrics["isi_peak_count"]),
        "avg_firing_rate_Hz": float(fig_metrics["fr_hz"]) if np.isfinite(fig_metrics["fr_hz"]) else np.nan,
        "n_spikes": int(fig_metrics["n_spikes"]),
        "figure_path": str(fig_path)
    }

# ---------------------- top-level: read input Excel & write outputs ----------------------

def kilosort_postprocess(excel_path):
    """
    Reads an input Excel file where each column describes one session:
      row 0: session_name (str)
      row 1: results_dir  (str)
      rows 2+: cluster IDs (ints)
    For each session:
      - Writes {results_dir}/{session_name}.xlsx (one sheet named {session_name})
        with per-cluster blocks separated by blank columns.
      - Writes/updates {results_dir}/best_channels.xlsx (as before).
      - Saves figures into {results_dir}/figures/{session_name}/
      - Saves timestamps into {results_dir}/timestamps/{cluster_id}.mat
    """
    df = pd.read_excel(excel_path, header=None)
    best_rows_by_dir = defaultdict(list)            # for best_channels.xlsx
    session_blocks = defaultdict(list)              # key: (results_dir, session_name) -> list of cluster metrics dicts

    for col in df.columns:
        session_name = str(df.iloc[0, col])
        results_path = str(df.iloc[1, col])
        cluster_ids = df.iloc[2:, col].dropna()

        for cluster_id in cluster_ids:
            cluster_id = int(cluster_id)
            print(f"Processing {session_name}, Cluster {cluster_id}")
            try:
                metrics = process_cluster(cluster_id, session_name, results_path)
                # keep best_channels per results_dir
                best_rows_by_dir[Path(results_path)].append({
                    "session": metrics["session"],
                    "cluster_id": metrics["cluster_id"],
                    "best_channel": metrics["best_channel"]
                })
                # accumulate for the per-session sheet
                session_blocks[(results_path, session_name)].append(metrics)
            except Exception as e:
                print(f"Error processing {session_name} Cluster {cluster_id}: {e}")

    # Write best_channels.xlsx per results_dir (unchanged behavior)
    for rdir, rows in best_rows_by_dir.items():
        if not rows:
            continue
        out_df = pd.DataFrame(rows).sort_values(["session", "cluster_id"]).reset_index(drop=True)
        out_df.to_excel(Path(rdir) / "best_channels.xlsx", index=False)

    # Write one workbook per session, with one sheet named after the session
    for (rdir, sname), rows in session_blocks.items():
        if not rows:
            continue

        # Cluster block layout: two columns (Metric / Value) per cluster + two blank spacer columns
        metric_order = [
            ("Cluster ID", "cluster_id"),
            ("Best channel", "best_channel"),
            ("Peak time (µs)", "peak_time_us"),
            ("Trough time (µs)", "trough_time_us"),
            ("Peak→Trough Δt (µs)", "peak_to_trough_us"),
            ("Peak (µV)", "peak_uV"),
            ("Trough (µV)", "trough_uV"),
            ("ISI peak center (µs)", "isi_peak_center_us"),
            ("ISI peak count", "isi_peak_count"),
            ("Avg firing rate (Hz)", "avg_firing_rate_Hz"),
            ("Spike count (n)", "n_spikes"),
            ("Figure path", "figure_path"),
        ]

        frames = []
        for r in rows:
            labels = [m[0] for m in metric_order]
            values = [r[m[1]] for m in [m for m in metric_order]]
            frames.append(pd.DataFrame({
                f"Metric (Cluster {r['cluster_id']})": labels,
                f"Value (Cluster {r['cluster_id']})": values
            }))
            # add two empty spacer columns
            frames.append(pd.DataFrame({"": ["" for _ in labels], "  ": ["" for _ in labels]}))

        wide_df = pd.concat(frames, axis=1)

        out_xlsx = Path(rdir) / f"{sname}.xlsx"
        with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
            wide_df.to_excel(writer, sheet_name=sname, index=False)

    print("Done.")


In [7]:
feedersheet_dir= Path(r"D:\Data\PTEN Project\feedersheets\single waveforms.xlsx")
kilosort_postprocess(feedersheet_dir)

Processing m6s8, Cluster 0
Processing m6s8, Cluster 2
Processing m6s8, Cluster 4
Processing m6s8, Cluster 12
Processing m6s8, Cluster 13
Processing m6s8, Cluster 19
Processing m6s8, Cluster 20
Processing m6s8, Cluster 36
Processing m6s8, Cluster 48
Processing m6s8, Cluster 52
Processing m6s8, Cluster 59
Processing m6s8, Cluster 60
Processing m6s8, Cluster 63
Processing m6s8, Cluster 64
Processing m6s8, Cluster 69
Processing m6s8, Cluster 71
Processing m6s8, Cluster 73
Done.
